<h1 style="text-align:center;">
Modeling Subscription Likelihood From Player Experience and Playtime
</h1>


Video game developers increasignly rely on structured analytics to better understand player behaviour and experience, using these insights to strengthen player engagement and reinforce major revenue streams. With a growing prominence in subscription-based gaming models, these developers use data-driven player metrics to forecast player retention to identify which players are most inclined to retain, or activate, a subscription to a particular game. By identifying which behaviours or variables are responsible for player subscription, developers are able to tailor and improve game design to encourage long-term engagement and player loyalty.

The project aims to answer the following question: **To what extent can player experience and total active playtime predict a player's subscription status?** From the **`players.csv`** data frame, we aimed to explore which variables influence player subscription status, and whether simple predictive models can be used to forecast future subscription outcomes with meaningful accuracy. 

The **`players.csv`** dataframe contains records of player sessional activity - it contains 196 observations (rows) and 9 variables (columns), where each row represents a single player. The dataframe includes player experience level, played hours, and subsctiption status, as well as other columns insignificant to our predictive model. The detailed description of the dataframe is as follows:
- **experience**: a categorical measure of player proficiency - includes:
**beginner,**
**amateur,**
**regular,**
**pro,**
**and**
**veteran.**
- **subscribe**: a categorical variable indicating whether a player has an active subscription - **includes boolean (True/False).**
- **hashedEmail**: a nominal measure to record player emails.
- **played_hours**: a temporal variable recording individual player session length.
- **name**: a nominal variable recording player name.
- **gender**: a nominal variable recording player gender.
- **age**: a nominal variable recording player age.
- **individualId**: N/A.
- **organizationName**: N/A.
    

In order to begin our analysis, we will load the necessary libraries, read in our data from the URL, and examine our dataset.

In [17]:
import altair as alt
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.compose import make_column_transformer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
url = "https://drive.google.com/uc?export=download&id=1Mw9vW0hjTJwRWx0bDXiSpYsO3gKogaPz"
players = pd.read_csv(url)
players.sample(10)

,experience,subscribe,hashedEmail,played_hours,name,gender,age,individualId,organizationName
29,Veteran,False,951e54f7376e2b2f0915e9e3646c701af4a2fe839385b1...,0.1,Vivienne,Male,18,NaN,NaN
146,Pro,True,5669c0f4b50dbfe3e2f851e4bb89fa43c55cd1f71ba362...,0.0,Padma,Non-binary,25,NaN,NaN
156,Amateur,True,8af6e04a9067b009303e36cf39f7e7d53f9473e74b4633...,0.1,Zelda,Male,23,NaN,NaN
35,Veteran,True,5a340c0e3d1aa3e579efc625bd3e5bca7fc25f7115b68e...,0.4,Zoe,Male,20,NaN,NaN
79,Amateur,True,dbb20960cd4db4900dde7832e349dae46f7688583c8138...,0.0,Jia,Female,26,NaN,NaN
44,Veteran,True,8d2eed1f399e0d77cebb8fcc48ed19ad2fa8e3bb3fa683...,2.2,Cyrus,Male,24,NaN,NaN
24,Amateur,True,119f01b9877fc5ea0073d05602a353b91c4b48e4cf02f4...,0.7,Hugo,Female,21,NaN,NaN
27,Veteran,False,f8acd26a5e452b682b9f8b20108caef1ee2e745efe08e9...,0.0,Finn,Male,23,NaN,NaN
58,Regular,True,f2826fb8dbce4d450348f99cb27ade184b713998d96797...,3.6,Zane,Male,10,NaN,NaN
126,Beginner,True,d51a8c57269fe347b4b7760ec29c420832cb36a11c4756...,0.7,Amelie,Female,24,NaN,NaN


Next, we will select only the columns we wish to perform analysis on: 'subscribe', 'played_hours', and 'age'.

In [18]:
player_selected = players[["subscribe","played_hours","age"]]
player_selected

,subscribe,played_hours,age
0,True,30.3,9
1,True,3.8,17
2,False,0.0,17
3,True,0.7,21
4,True,0.1,21
...,...,...,...
191,True,0.0,17
192,False,0.3,22
193,False,0.0,17
194,False,2.3,17


In order for our analysis to be reproducible, we will set a random seed.

In [19]:
np.random.seed(1)

Now, we will split the train set and test set.

In [7]:
player_train, player_test = train_test_split(
    player_selected, train_size=0.75, stratify=player_selected["subscribe"]
)
player_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 147 entries, 95 to 61
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   subscribe     147 non-null    bool   
 1   played_hours  147 non-null    float64
 2   age           147 non-null    int64  
dtypes: bool(1), float64(1), int64(1)
memory usage: 3.6 KB


And standardize the data.

In [8]:
player_preprocessor = make_column_transformer(
    (StandardScaler(), ["played_hours", "age"]),
)

Next, we train the classifier with the k = 3 and form our pipeline.

In [9]:
X = player_train[["played_hours", "age"]]
y = player_train["subscribe"]
knn = KNeighborsClassifier(n_neighbors=3)
knn_pipeline = make_pipeline(player_preprocessor, knn)
knn_pipeline.fit(X, y)

knn_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('standardscaler',
                                                  StandardScaler(),
                                                  ['played_hours', 'age'])])),
                ('kneighborsclassifier', KNeighborsClassifier(n_neighbors=3))])

Now, we can predict the test set with the model and test the score.

In [10]:
player_test["predicted"] = knn_pipeline.predict(player_test[["played_hours", "age"]])
player_test[["subscribe", "predicted"]]
knn_pipeline.score(
    player_test[["played_hours", "age"]],
    player_test["subscribe"]
)

0.7346938775510204

And perform cross validation.

In [12]:
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GridSearchCV

knn = KNeighborsClassifier()
player_tune_pipe = make_pipeline(player_preprocessor, knn)

parameter_grid = {
    "kneighborsclassifier__n_neighbors": range(2, 20, 1),
}

player_tune_grid = GridSearchCV(
    estimator=player_tune_pipe,
    param_grid=parameter_grid,
    cv=10
)

player_tune_grid.fit(
    player_train[["played_hours", "age"]],
    player_train["subscribe"]
)

accuracies_grid = pd.DataFrame(player_tune_grid.cv_results_)
accuracies_grid.info()

accuracies_grid["sem_test_score"] = accuracies_grid["std_test_score"] / 10**(1/2)
accuracies_grid = (
    accuracies_grid[[
        "param_kneighborsclassifier__n_neighbors",
        "mean_test_score",
        "sem_test_score"
    ]]
    .rename(columns={"param_kneighborsclassifier__n_neighbors": "n_neighbors"})
)
accuracies_grid

accuracy_vs_k = alt.Chart(accuracies_grid).mark_line(point=True).encode(
    x=alt.X("n_neighbors").title("Neighbors"),
    y=alt.Y("mean_test_score")
        .scale(zero=False)
        .title("Accuracy estimate")
)

accuracy_vs_k

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 19 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   mean_fit_time                            18 non-null     float64
 1   std_fit_time                             18 non-null     float64
 2   mean_score_time                          18 non-null     float64
 3   std_score_time                           18 non-null     float64
 4   param_kneighborsclassifier__n_neighbors  18 non-null     int64  
 5   params                                   18 non-null     object 
 6   split0_test_score                        18 non-null     float64
 7   split1_test_score                        18 non-null     float64
 8   split2_test_score                        18 non-null     float64
 9   split3_test_score                        18 non-null     float64
 10  split4_test_score                        18 non-null

alt.Chart(...)

From the plot we can see k = 8 might be the best classifier, and as such, we will train the model with k = 8 

In [13]:
X = players[["age", "played_hours"]]      
y = players["subscribe"]                  
exp = players["experience"]               

# Train/test split
X_train, X_test, y_train, y_test, exp_train, exp_test = train_test_split(
    X, y, exp,
    test_size=0.25,
    stratify=y,          
    random_state=2025,
)

knn_pipe = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=8)
)

knn_pipe.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('kneighborsclassifier', KNeighborsClassifier(n_neighbors=8))])

Now, we can take a look at our new test data.

In [15]:
y_pred = knn_pipe.predict(X_test)

results = pd.DataFrame({
    "experience": exp_test.values,
    "y_true": y_test.values,
    "y_pred": y_pred,
})

results["correct"] = results["y_true"] == results["y_pred"]
acc_by_exp = (
    results
    .groupby("experience")["correct"]
    .mean()
    .reset_index()
    .rename(columns={"correct": "accuracy"})
)

acc_by_exp

,experience,accuracy
0,Amateur,0.642857
1,Beginner,0.555556
2,Pro,1.000000
3,Regular,0.888889
4,Veteran,0.714286


Lastly, we can perform a visualization of our data, plotting how accuracy changes across different experience categories.

In [20]:
acc_plot = alt.Chart(acc_by_exp).mark_bar().encode(
    x=alt.X("experience:N", title="Experience level"),
    y=alt.Y("accuracy:Q", title="Prediction accuracy"),
    tooltip=[
        "experience",
        alt.Tooltip("accuracy:Q", title="Accuracy", format=".2f")
    ]
).properties(
    title="KNN (k = 8) accuracy by experience level"
)
acc_plot

alt.Chart(...)

## Discussion

### (1) What We Found
Using only **age** and **played_hours** to predict whether a player subscribes (binary), a KNN pipeline with standardization and CV-based model selection suggested **k ≈ 8** (see Accuracy vs. Neighbors plot). With this setting, the simple train/test split achieved a modest overall accuracy (~0.57 on an earlier hold-out; cross-validated estimates peaked ~0.72). When we broke test performance down by the experience groups (not used as a predictor, only for post-hoc analysis), prediction accuracy varied substantially:
- **Beginner ≈ 0.56, Amateur ≈ 0.64**
- **Regular ≈ 0.89, Veteran ≈ 0.71**
- **Pro = 1.00** (very small sample likely)

Two qualitative insights emerge:
- Engagement intensity matters: players with more hours tend to be predicted as subscribers more often, consistent with the idea that playtime is a proxy for interest/commitment.
- Behavior is more stereotyped among experienced players: prediction is easiest for **Regular/Pro/Veteran** groups and hardest for **Beginners**. This suggests clearer, more consistent engagement patterns among experienced players, while novices behave more heterogeneously, for example, some try once and leave; some ramp up quickly).

### (2) Did Our Results Align With Our Initial Predictions?
Broadly **yes**,the result is what we expected to find. We expected more experienced and more active players to be more likely subscribers, and the model’s class-wise accuracies align with that narrative.
But there still have two surprises:
- Perfect accuracy for “Pro” almost certainly reflects very small n and/or an easy boundary for that subgroup; it should not be over-interpreted.
- The overall accuracy is only moderate when restricted to two numeric features. That indicates meaningful information is missing from the feature set.

### (3) The Impacts of These Findings
Although with modest overall accuracy, the patterns are useful for recruiting and resource planning on the research server:
- Targeting: outreach that specifically reaches Regular/Pro/Veteran players (e.g., communities, leaderboards, Discords, or events frequented by those tiers) is likely to yield higher subscription rates.
- Capacity planning: if subscription is used as a proxy for continuing participation, experienced cohorts should weigh more heavily when forecasting steady demand.
- Onboarding: Since Beginners have uncertainty and are the most difficult to predict/convert, investing in early experience interventions(such as tutorials, short tasks, nudges to return) may enhance conversion in areas where the current model struggles.

### (4) Limitations
- Small dataset (n≈196) and uneven subgroup sizes make estimates noisy, especially for "Pro".
- We only used two predictors. Experience was not included as a model feature (just for stratified evaluation). Many behavioral signals are absent.
- The single random training/testing split in the final stratified analysis may increase the variance.

### (5) Future questions
- **Gender** may have potential relations but now it not clear, we can do some further analysis (for example, stratify by self-reported gender and test interactions) to expore the specific relations in the future.
- We can **compare different models** and explore if they outperform KNN on generalization and interpretability.
- Continute explore the **variables we didn't include**. Some additional predictors that might explain uptake (such as time-of-day/week activity patterns, session streaks/retention, social features like party play, exposure to newsletter prompts), then re-evaluate performance.
- Understanding **who** subscribes helps developers design targeted nudges. We can compare subscription rates and feature importances across groups (such as gender, age bands, region) and check interaction effects. Any analysis should include fairness/bias checks and report confidence intervals.

### (6) Summary
Even a simple and fully replicable KNN classification demonstrates that the combination of engagement (played_hours) coupled with player maturity (experience) can provide information about subscription possibilities. However, before deploying reliable decision support for recruitment or capability planning, more features, better validation and calibration, and interpretable models are needed. Therefore, the next iteration should expand the feature space, conduct benchmark tests on multiple models, and quantify uncertainty so that stakeholders can act with confidence.